# Fase 1 — Definición del proyecto y entorno reproducible

**Proyecto:** Sobreduración en la titulación de la educación superior chilena (2020–2025)

**Curso:** «Nombre del curso» («Código»)  ·  **Docente:** «Nombre del docente»
**Equipo:** «Integrante 1», «Integrante 2», «Integrante 3»  ·  **Grupo:** «N»
**Fecha:** «dd/mm/aaaa»  ·  **Repositorio:** «URL del repositorio GitHub»

---

Este notebook corresponde **exclusivamente al alcance de la Fase 1**: definir la
problemática, formular objetivos, dejar operativo el entorno reproducible y
verificar que los insumos y el código base funcionan. La obtención, limpieza y
transformación del conjunto de datos se desarrollan en los notebooks de la
Fase 2.

## 1. Contexto y problemática

El Servicio de Información de Educación Superior (SIES) del Ministerio de
Educación publica anualmente la base de **titulados de educación superior por
estudiante**, con un registro por título obtenido. Entre 2020 y 2025 esa base
acumula **1.710.167 registros**, lo que la convierte en la fotografía más
completa disponible sobre el final de la trayectoria formativa en Chile.

El sistema chileno declara para cada programa una **duración teórica** (el plan
de estudios comprometido con el estudiante), pero no publica de manera directa
cuánto tardan efectivamente los estudiantes en titularse. La diferencia entre
ambas magnitudes —la **sobreduración**— tiene consecuencias concretas:
prolonga el gasto de las familias, extiende el uso de becas y créditos
estatales, retrasa la entrada al mercado laboral y desajusta la planificación
de vacantes de las instituciones.

**El problema es que esa brecha no está publicada: hay que construirla.** La
base entrega los insumos (año y semestre de ingreso a la carrera de origen,
fecha exacta de obtención del título y duración teórica del plan), pero
distribuidos en seis archivos anuales de ~955 MB en total, con codificaciones
centinela en lugar de valores nulos y sin ninguna variable derivada. Sin un
pipeline reproducible de procesamiento, la pregunta no puede responderse.

## 2. Preguntas centrales del análisis

1. ¿Cuánto excede la duración real de una carrera de pregrado a su duración
   teórica, y qué proporción de estudiantes se titula efectivamente dentro del
   plazo comprometido?
2. ¿Cómo varía esa sobreduración según **área de conocimiento**, **tipo de
   institución** (universidad, instituto profesional, centro de formación
   técnica) y **región de la sede**?
3. ¿Existe una **brecha de género** en los tiempos de titulación, y cómo se
   relaciona con la composición por sexo de cada área?
4. ¿La expansión de la **modalidad no presencial** entre 2020 y 2025 se asocia
   a trayectorias más breves o más largas, y a qué perfil de estudiante
   corresponde?
5. ¿Qué efecto muestran los datos en el año **2020**, marcado por la
   interrupción de la actividad presencial?

## 3. Objetivos

### Objetivo general

Construir, a partir de los registros administrativos de titulados de educación
superior del SIES (2020–2025), un conjunto de datos analítico y reproducible que
permita **cuantificar y caracterizar la sobreduración de las carreras de
pregrado en Chile**, identificando las dimensiones institucionales,
territoriales, de género y de modalidad asociadas al rezago en la titulación.

### Objetivos específicos

| N.º | Objetivo específico | Fase |
|-----|---------------------|------|
| OE1 | Establecer un entorno de trabajo reproducible (entorno virtual, gestión de dependencias, control de versiones y estructura modular del código). | F1 |
| OE2 | Consolidar los seis archivos anuales en un único conjunto, verificando su integridad contra las cifras oficiales publicadas por el SIES. | F2 |
| OE3 | Explorar la estructura, distribuciones y problemas de calidad del conjunto consolidado. | F2 |
| OE4 | Depurar el conjunto: tratar codificaciones centinela, valores faltantes, duplicados y registros atípicos, documentando cada decisión. | F2 |
| OE5 | Derivar las variables analíticas del proyecto: duración real, sobreduración, índice de duración, edad de titulación y categoría de rezago. | F2 |
| OE6 | Validar técnicamente el conjunto resultante mediante un motor de reglas y pruebas automatizadas. | F2 |
| OE7 | Modelar la sobreduración y evaluar su capacidad predictiva. | F3 |
| OE8 | Comunicar los hallazgos en un reporte analítico con visualizaciones efectivas. | F4 |

## 4. Alcance y supuestos

### Qué aborda este proyecto

- Titulados de **pregrado** entre 2020 y 2025 (seis procesos anuales completos).
- Cobertura nacional: 16 regiones, los tres tipos de institución y las diez
  áreas de conocimiento de la clasificación MINEDUC.
- Variables demográficas (sexo, edad), institucionales (tipo, jornada,
  modalidad), territoriales (región, provincia, comuna) y curriculares (área,
  nivel, duración teórica).

### Qué queda fuera

- **Posgrado y postítulo** (398.391 registros): sus planes tienen duraciones
  heterogéneas y no comparables con las de una carrera regular, por lo que
  mezclarlos distorsionaría la medición de sobreduración.
- **Deserción y retención**: la base solo contiene a quienes se titularon; no
  permite observar a quienes abandonaron.
- **Series anteriores a 2020**: el SIES publica desde 2007, pero la ventana se
  acota a seis años para mantener homogéneo el esquema de variables (la
  clasificación CINE-F 2013 se incorpora recién en 2021).
- **Inferencia causal**: el diseño es descriptivo y correlacional; las
  diferencias observadas entre grupos no se interpretan como efectos causales.

### Supuestos

| # | Supuesto | Implicancia |
|---|----------|-------------|
| S1 | El año y semestre de ingreso a la **carrera de origen** representan el inicio real de la trayectoria. | Es la variable que el propio SIES indica usar como referencia de ingreso. |
| S2 | Un título obtenido hasta julio se imputa al primer semestre y desde agosto al segundo. | Los datos traen fecha exacta de titulación, pero no el semestre académico; se requiere una convención explícita. |
| S3 | La duración teórica informada (`dur_total_carr`) corresponde al plan vigente del estudiante. | Cambios de plan durante la trayectoria no son observables en la base. |
| S4 | Los registros sin insumos para reconstruir la duración se descartan en lugar de imputarse. | Imputar el año de ingreso equivaldría a inventar la variable que se busca medir. |
| S5 | La base cubre a la totalidad de los titulados informados por las instituciones. | El SIES documenta solo dos omisiones institucionales, ambas anteriores a 2012. |

## 5. Del mapa conceptual técnico a la implementación

El mapa conceptual elaborado en la actividad formativa define cinco componentes.
La tabla siguiente indica dónde se materializa cada uno en este repositorio.

| Componente del mapa | Materialización | Fase |
|---------------------|-----------------|------|
| Entorno reproducible | `.venv/` + `requirements.txt` + `config.describir_entorno()` | F1 |
| Control de versiones | Repositorio Git con ramas por fase e historial de commits descriptivos | F1–F4 |
| Estructura modular | Paquete `src/` con seis módulos de responsabilidad única | F1 |
| Obtención de datos | `src/ingesta.py` — lectura por bloques y consolidación en Parquet | F2 |
| Exploración (EDA) | `F2/F2_1_Obtencion_Exploracion.ipynb` | F2 |
| Limpieza y transformación | `src/limpieza.py`, `src/transformacion.py` | F2 |
| Validación técnica | `src/validacion.py` (motor de reglas) + `tests/` (pytest) | F2 |
| Visualización | `src/viz.py` con estilo unificado | F2–F4 |
| Modelación | Proyectado: predicción de la sobreduración | F3 |
| Comunicación de hallazgos | Proyectado: reporte analítico final | F4 |

## 6. Verificación del entorno reproducible

Primera evidencia de reproducibilidad: la versión exacta de Python y de cada
librería con la que se produjeron los resultados queda registrada en la salida
ejecutada del notebook.

In [1]:
# Celda de arranque: hace importable el paquete `src` sin instalar el proyecto
# y funciona igual si el notebook se abre desde la raiz o desde su subcarpeta.
import sys
from pathlib import Path

RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

from src import config

print("Raiz del proyecto:", RAIZ)
for clave, valor in config.describir_entorno().items():
    print(f"  {clave:12s} {valor}")

Raiz del proyecto: D:\IA Development\proyecto-titulados


  python       3.14.7
  plataforma   Windows-11-10.0.26200-SP0
  ejecutable   D:\IA Development\proyecto-titulados\.venv\Scripts\python.exe
  numpy        2.5.3
  pandas       3.0.5
  matplotlib   3.11.1
  seaborn      0.13.2
  pyarrow      25.0.1


### 6.1 Correspondencia con el archivo de dependencias

Se comprueba que todo lo declarado en `requirements.txt` esté efectivamente
instalado en el entorno virtual activo. Si algo falta, la celda lo informa en
lugar de fallar más adelante en medio del pipeline.

In [2]:
import importlib.metadata as md_meta
import re

requisitos = (RAIZ / "requirements.txt").read_text(encoding="utf-8").splitlines()
paquetes = [
    re.split(r"[=<>~!]", linea)[0].strip()
    for linea in requisitos
    if linea.strip() and not linea.strip().startswith("#")
]

filas = []
for paquete in paquetes:
    try:
        version = md_meta.version(paquete)
        estado = "instalado"
    except md_meta.PackageNotFoundError:
        version, estado = "-", "FALTANTE"
    filas.append({"paquete": paquete, "version_instalada": version, "estado": estado})

estado_entorno = pd.DataFrame(filas)
print(estado_entorno.to_string(index=False))
assert (estado_entorno["estado"] == "instalado").all(), "Faltan dependencias declaradas"
print("\nEntorno completo: todas las dependencias declaradas estan instaladas.")

   paquete version_instalada    estado
     numpy             2.5.3 instalado
    pandas             3.0.5 instalado
   pyarrow            25.0.1 instalado
matplotlib            3.11.1 instalado
   seaborn            0.13.2 instalado
jupyterlab             4.6.3 instalado
  notebook             7.6.2 instalado
 ipykernel             7.3.0 instalado
  nbformat            5.11.1 instalado
 nbconvert            7.17.1 instalado
    pytest             9.1.1 instalado

Entorno completo: todas las dependencias declaradas estan instaladas.


## 7. Estructura del repositorio

La organización separa datos, código, notebooks por fase, pruebas y salidas.
Ninguna ruta absoluta aparece en los notebooks: todo se resuelve desde
`src/config.py`.

In [3]:
def arbol(directorio, prefijo="", ignorar=(".git", ".venv", "__pycache__", ".ipynb_checkpoints")):
    """Imprime el arbol de carpetas del proyecto, omitiendo directorios tecnicos."""
    entradas = sorted(
        [e for e in directorio.iterdir() if e.name not in ignorar],
        key=lambda e: (e.is_file(), e.name.lower()),
    )
    for i, entrada in enumerate(entradas):
        ultimo = i == len(entradas) - 1
        rama = "\\-- " if ultimo else "|-- "
        if entrada.is_dir():
            print(f"{prefijo}{rama}{entrada.name}/")
            arbol(entrada, prefijo + ("    " if ultimo else "|   "), ignorar)
        else:
            tam = entrada.stat().st_size
            unidad = f"{tam/1024**2:.1f} MB" if tam > 1024**2 else f"{tam/1024:.0f} KB"
            print(f"{prefijo}{rama}{entrada.name}  ({unidad})")


print(f"{RAIZ.name}/")
arbol(RAIZ)

proyecto-titulados/
|-- .pytest_cache/
|   |-- v/
|   |   \-- cache/
|   |       |-- lastfailed  (0 KB)
|   |       \-- nodeids  (1 KB)
|   |-- .gitignore  (0 KB)
|   |-- CACHEDIR.TAG  (0 KB)
|   \-- README.md  (0 KB)
|-- data/
|   |-- interim/
|   |   \-- titulados_2020_2025_consolidado.parquet  (37.5 MB)
|   |-- processed/
|   |   |-- muestra_titulados_5000.csv  (2.2 MB)
|   |   \-- titulados_pregrado_analitico.parquet  (33.9 MB)
|   \-- raw/
|       |-- 20260817_Titulados_Ed_Superior_2020_WEB.csv  (109.8 MB)
|       |-- 20260817_Titulados_Ed_Superior_2021_WEB.csv  (149.4 MB)
|       |-- 20260817_Titulados_Ed_Superior_2022_WEB.csv  (154.7 MB)
|       |-- 20260817_Titulados_Ed_Superior_2023_WEB.csv  (159.5 MB)
|       |-- 20260817_Titulados_Ed_Superior_2024_WEB.csv  (162.4 MB)
|       \-- 20260817_Titulados_Ed_Superior_2025_WEB.csv  (175.6 MB)
|-- docs/
|   \-- ER titulados Ed.Superior 2007 - 2025, WEB.pdf  (219 KB)
|-- F1/
|   \-- F1_Definición.ipynb  (49 KB)
|-- F2/
|   |-- F2_1_Obt

## 8. Disponibilidad y trazabilidad de los datos de origen

Los seis CSV originales pesan cerca de 955 MB y **no se versionan en GitHub**
(el límite por archivo de la plataforma es de 100 MB). El repositorio incluye el
código que los localiza y procesa; la ubicación puede redefinirse con la
variable de entorno `TITULADOS_RAW_DIR`.

In [4]:
from src import ingesta

archivos = ingesta.listar_archivos_raw()

inventario = pd.DataFrame(
    [
        {
            "anio": anio,
            "archivo": ruta.name,
            "tamano_mb": round(ruta.stat().st_size / 1024**2, 1),
            "filas_oficiales_sies": config.FILAS_OFICIALES_SIES[anio],
        }
        for anio, ruta in archivos.items()
    ]
)
print(inventario.to_string(index=False))
print(f"\nTotal: {inventario['tamano_mb'].sum():,.0f} MB y "
      f"{inventario['filas_oficiales_sies'].sum():,} registros esperados.")

 anio                                     archivo  tamano_mb  filas_oficiales_sies
 2020 20260817_Titulados_Ed_Superior_2020_WEB.csv      109.8                203598
 2021 20260817_Titulados_Ed_Superior_2021_WEB.csv      149.4                281491
 2022 20260817_Titulados_Ed_Superior_2022_WEB.csv      154.7                291460
 2023 20260817_Titulados_Ed_Superior_2023_WEB.csv      159.5                299893
 2024 20260817_Titulados_Ed_Superior_2024_WEB.csv      162.4                304727
 2025 20260817_Titulados_Ed_Superior_2025_WEB.csv      175.6                328998

Total: 911 MB y 1,710,167 registros esperados.


## 9. Primera lectura exploratoria (muestra)

En Fase 1 solo se comprueba que los archivos son legibles y que el esquema
coincide con lo declarado en el documento oficial del SIES. La carga completa
corresponde a la Fase 2.

In [5]:
muestra = pd.read_csv(
    archivos[2025],
    sep=config.SEPARADOR,
    encoding=config.ENCODING,
    nrows=1_000,
    dtype=str,
)

print(f"Dimensiones de la muestra: {muestra.shape[0]} filas x {muestra.shape[1]} columnas")
print(f"Columnas declaradas en el esquema del proyecto: {len(config.COLUMNAS_USADAS)}")
print(f"Columnas descartadas y justificadas: {len(config.COLUMNAS_DESCARTADAS)}")

faltan = set(config.COLUMNAS_USADAS) - set(muestra.columns)
assert not faltan, f"El archivo no contiene las columnas esperadas: {faltan}"
print("\nEsquema verificado: todas las columnas requeridas estan presentes.")

muestra[["cat_periodo", "gen_alu", "anio_ing_carr_ori", "fecha_obtencion_titulo",
         "nomb_carrera", "dur_total_carr", "region_sede", "modalidad"]].head()

Dimensiones de la muestra: 1000 filas x 40 columnas
Columnas declaradas en el esquema del proyecto: 30
Columnas descartadas y justificadas: 10

Esquema verificado: todas las columnas requeridas estan presentes.


,cat_periodo,gen_alu,anio_ing_carr_ori,fecha_obtencion_titulo,nomb_carrera,dur_total_carr,region_sede,modalidad
0,2025,1,2022,20250328,MAGISTER EN DATA SCIENCE,3,Metropolitana,Presencial
1,2025,1,2015,20251105,INGENIERIA CIVIL INDUSTRIAL,7,Valparaíso,Presencial
2,2025,2,2022,20250613,TECNICO EN ENFERMERIA,5,Metropolitana,Presencial
3,2025,2,2022,20250930,TECNICO EN ODONTOLOGIA,5,Valparaíso,Presencial
4,2025,1,2021,20250315,ANALISTA PROGRAMADOR COMPUTACIONAL,5,Metropolitana,Presencial


### 9.1 Justificación de las columnas descartadas

De las 40 columnas del archivo original se utilizan 30. El descarte no es
arbitrario: cada exclusión responde a redundancia o a irrelevancia frente a las
preguntas planteadas, y queda documentada en el código.

In [6]:
descartes = pd.DataFrame(
    config.COLUMNAS_DESCARTADAS.items(), columns=["columna", "motivo_del_descarte"]
)
print(descartes.to_string(index=False))

          columna                                                      motivo_del_descarte
    nombre_titulo        Texto libre de alta cardinalidad; no aporta al analisis agregado.
     nombre_grado              Vacio en carreras tecnicas; redundante con nivel_carrera_1.
      tipo_inst_3        Desagregacion de tipo_inst_2 sin uso en las preguntas planteadas.
anio_ing_carr_act       Refiere a convalidacion; el SIES indica usar la carrera de origen.
 sem_ing_carr_act                                                           Idem anterior.
         cod_sede El analisis territorial se realiza a nivel de comuna/region, no de sede.
        nomb_sede                                                           Idem anterior.
          version                   Metadato administrativo del plan, sin valor analitico.
   cine_f_97_area                     Clasificacion CINE-F 1997, superada por CINE-F 2013.
cine_f_97_subarea                                                           Idem anterior.

## 10. Código base del proyecto: modularidad y programación orientada a objetos

El código no vive en los notebooks: reside en el paquete `src/`, y los notebooks
lo invocan. Esto evita duplicar lógica entre fases y permite probar cada función
de manera aislada.

Dos componentes están implementados con clases porque necesitan **mantener
estado** entre llamadas: la bitácora de limpieza (acumula los pasos aplicados) y
el validador (acumula reglas y resultados).

In [7]:
import inspect

from src import limpieza, transformacion, validacion, viz

modulos = {
    "config": config,
    "ingesta": ingesta,
    "limpieza": limpieza,
    "transformacion": transformacion,
    "validacion": validacion,
    "viz": viz,
}

resumen_modulos = pd.DataFrame(
    [
        {
            "modulo": f"src/{nombre}.py",
            "funciones": len(
                [f for f, o in inspect.getmembers(mod, inspect.isfunction)
                 if o.__module__ == mod.__name__ and not f.startswith("_")]
            ),
            "clases": len(
                [c for c, o in inspect.getmembers(mod, inspect.isclass)
                 if o.__module__ == mod.__name__]
            ),
            "proposito": (mod.__doc__ or "").strip().split("\n")[0],
        }
        for nombre, mod in modulos.items()
    ]
)
print(resumen_modulos.to_string(index=False))

               modulo  funciones  clases                                                              proposito
        src/config.py          1       0                                    Configuracion central del proyecto.
       src/ingesta.py          4       1            Obtencion y consolidacion de los archivos anuales del SIES.
      src/limpieza.py          7       2 Depuracion del consolidado: sentinelas, faltantes, duplicados y tipos.
src/transformacion.py         11       0                   Derivacion de las variables analiticas del proyecto.
    src/validacion.py          1       2                             Motor de validacion del dataset analitico.
           src/viz.py          9       0                       Funciones de visualizacion con estilo unificado.


### 10.1 Demostración de la bitácora de limpieza (clase `BitacoraLimpieza`)

Cada paso del pipeline registra cuántas filas entraron, cuántas salieron y por
qué. El objeto se convierte en una tabla auditable que se exporta como evidencia.

In [8]:
bitacora_demo = limpieza.BitacoraLimpieza()
bitacora_demo.registrar("carga", "Consolidado de ejemplo", 1000, 1000)
bitacora_demo.registrar("filtrar_nivel", "Solo pregrado", 1000, 780)
bitacora_demo.registrar("eliminar_duplicados", "Estudiante + carrera + fecha", 780, 775)

print(bitacora_demo.a_dataframe().to_string(index=False))
print(f"\nPasos registrados: {len(bitacora_demo)}")

               paso                      detalle  filas_antes  filas_despues  filas_afectadas  porcentaje
              carga       Consolidado de ejemplo         1000           1000                0       0.000
      filtrar_nivel                Solo pregrado         1000            780              220      22.000
eliminar_duplicados Estudiante + carrera + fecha          780            775                5       0.641

Pasos registrados: 3


### 10.2 Demostración del motor de validación (clases `Regla` y `ValidadorDataset`)

Se construye un DataFrame mínimo de tres registros ficticios, se le aplican las
reglas del proyecto y se observa el veredicto. La ejecución sobre el conjunto
real se realiza en la Fase 2.

In [9]:
ejemplo = pd.DataFrame(
    [
        {"cat_periodo": anio, "mrun": 900 + i, "gen_alu": 1 + i % 2,
         "fec_nac_alu": 199603, "anio_ing_carr_ori": anio - 5,
         "sem_ing_carr_ori": 1, "fecha_obtencion_titulo": int(f"{anio}1215"),
         "dur_total_carr": 10, "nivel_global": "Pregrado"}
        for i, anio in enumerate(config.ANIOS)
    ]
)

ejemplo = transformacion.construir_dataset_analitico(ejemplo)
print(ejemplo[["cat_periodo", "duracion_real_sem", "dur_total_carr",
               "sobreduracion_sem", "categoria_rezago", "genero"]].to_string(index=False))

validador = validacion.validador_estandar()
reporte = validador.ejecutar(ejemplo)
print("\n" + reporte[["regla", "estado", "detalle"]].to_string(index=False))
print("\n" + validador.resumen())

 cat_periodo  duracion_real_sem  dur_total_carr  sobreduracion_sem categoria_rezago genero
        2020                 12              10                  2      Rezago leve Hombre
        2021                 12              10                  2      Rezago leve  Mujer
        2022                 12              10                  2      Rezago leve Hombre
        2023                 12              10                  2      Rezago leve  Mujer
        2024                 12              10                  2      Rezago leve Hombre
        2025                 12              10                  2      Rezago leve  Mujer

                    regla estado                                                                                  detalle
         dataset_no_vacio   PASA                                                         6 filas en el dataset analitico.
       columnas_derivadas   PASA                                                 Todas las variables derivadas present

### 10.3 Verificación manual del cálculo central

Antes de aplicar la fórmula a más de un millón de registros se comprueba a mano
sobre un caso conocido: una estudiante que ingresa el primer semestre de 2018 y
obtiene su título en diciembre de 2024 en un plan de 10 semestres.

In [10]:
caso = pd.DataFrame([{
    "cat_periodo": 2024, "mrun": 1, "gen_alu": 2, "fec_nac_alu": 199805,
    "anio_ing_carr_ori": 2018, "sem_ing_carr_ori": 1,
    "fecha_obtencion_titulo": 20241220, "dur_total_carr": 10,
    "nivel_global": "Pregrado",
}])

resultado = transformacion.construir_dataset_analitico(caso).iloc[0]

esperado = (2024 - 2018) * 2 + (2 - 1) + 1  # anios * 2 + diferencia de semestre + 1
print(f"Calculo manual : {esperado} semestres")
print(f"Calculo del codigo: {resultado['duracion_real_sem']} semestres")
print(f"Duracion teorica  : {resultado['dur_total_carr']} semestres")
print(f"Sobreduracion     : {resultado['sobreduracion_sem']} semestres "
      f"({resultado['categoria_rezago']})")
print(f"Indice de duracion: {resultado['indice_duracion']}")
print(f"Edad al titularse : {resultado['edad_titulacion']} anios")

assert resultado["duracion_real_sem"] == esperado
print("\nVerificacion superada: la implementacion coincide con el calculo manual.")

Calculo manual : 14 semestres
Calculo del codigo: 14 semestres
Duracion teorica  : 10 semestres
Sobreduracion     : 4 semestres (Rezago moderado)
Indice de duracion: 1.4
Edad al titularse : 26 anios

Verificacion superada: la implementacion coincide con el calculo manual.


## 11. Pruebas automatizadas del código base

La suite de `pytest` cubre casos normales, casos límite y excepciones. Se
ejecuta desde el notebook para dejar la evidencia dentro del propio documento.

In [11]:
import subprocess

resultado_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-q", "--no-header"],
    cwd=RAIZ, capture_output=True, text=True,
)
print(resultado_tests.stdout[-1500:])
print("Codigo de salida:", resultado_tests.returncode)

......................                                                   [100%]
22 passed in 1.77s

Codigo de salida: 0


## 12. Trazabilidad con el repositorio Git

El historial de commits evidencia el avance progresivo del proyecto y permite
vincular cada resultado con el estado exacto del código que lo produjo.

In [12]:
def git(*argumentos):
    """Ejecuta un comando git en la raiz del proyecto y devuelve su salida."""
    proceso = subprocess.run(
        ["git", *argumentos], cwd=RAIZ, capture_output=True, text=True
    )
    return proceso.stdout.strip() or proceso.stderr.strip()


print("Rama actual:", git("rev-parse", "--abbrev-ref", "HEAD"))
print("Commit actual:", git("rev-parse", "--short", "HEAD"))
print("\nUltimos commits:")
print(git("log", "--oneline", "-15"))
print("\nContribuciones por integrante:")
print(git("shortlog", "-sn", "--all"))

Rama actual: main
Commit actual: b113334

Ultimos commits:
b113334 docs(informe): agrega el informe tecnico de las Fases 1 y 2
82f72f3 style(viz): corrige la acentuacion de las etiquetas de los graficos
d64cac0 style(viz): rotula las figuras con formato numerico chileno
64c8e48 merge: integra la rama fase-2/pipeline-datos en main
86aa78d docs(f2): ejecuta los notebooks y versiona figuras, tablas y muestra
9a58f78 perf(limpieza): normaliza el catalogo de categorias en vez de fila por fila
d41cc30 fix(viz,transformacion): compatibilidad con los tipos anulables de pandas
1e0a197 docs(f2): notebooks de obtencion, limpieza, transformacion y validacion
0dbee5a docs(f1): notebook de definicion del proyecto y entorno reproducible
06cb784 feat(viz,pipeline): estilo unificado de figuras y orquestador ejecutable
6c0e566 test: cubre casos normales, limite y excepciones del pipeline
280fb6c feat(validacion): motor de reglas orientado a objetos
0e125ee feat(transformacion): reconstruye la duracion r

## 13. Cierre de la Fase 1 y proyección

**Estado alcanzado.** El entorno virtual está creado y sus dependencias
declaradas y verificadas; el código base está modularizado en seis módulos con
funciones documentadas y dos clases con estado; las pruebas automatizadas pasan;
los seis archivos de origen están localizados y su esquema validado contra el
documento oficial del SIES; y el repositorio Git registra el avance.

**Lo que sigue.**

| Fase | Contenido | Estado |
|------|-----------|--------|
| F2 | Consolidación, exploración, limpieza, transformación y validación del conjunto. | En esta entrega |
| F3 | Modelación de la sobreduración y evaluación de su capacidad predictiva. | Proyectado |
| F4 | Reporte analítico final con visualizaciones y conclusiones. | Proyectado |

> **Continuar en:** `F2/F2_1_Obtencion_Exploracion.ipynb`